<a href="https://colab.research.google.com/github/cras-lab/GenAI/blob/main/%EC%83%9D%EC%84%B1%EB%AA%A8%EB%8D%B8_ToyExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 임포트 한다.

In [1]:
import math
import numpy as np
import torch              # 파이토치를 활용
import torch.nn as nn     # 파이토치의 신경망 사용
import matplotlib.pyplot as plt

난수 생성의 시드를 42로 설정

In [ ]:
torch.manual_seed(42)

신경망, 실험을 위해 이번엔 2000개 수를 생성. 원래 예제 6개 숫자 포함

먼저 기존의 6개 숫자를 포함하기 이해 숫자를 생성한다.

In [3]:
# 기존 6개 숫자
original = torch.tensor(
    [5.2, 7.1, 8.8, 11.4, 12.7, 15.3]
)

실습용으로 추가로 2000개의 숫자를 추가한다. 두 개의 봉우리를 만들어 합칠것이므로, 동전을 던져 앞뒷면을 봐서 앞면이면 1)번 분포, 2)면 2번 분포 처럼 할 예정

In [4]:
n = 2000
group = torch.rand(n) < 0.5 # 동전 던지기 역할

torch.where(조건, A, B):  
조건이 참(True)인 위치에는 A에서 값을 가져오고,   
거짓(False)인 위치에는 B에서 값을 가져와 새로운 텐서 구성

In [5]:
extra = torch.where(
    group,                      # 50% 확률로 A나 B를 선택
    6.5 + 0.8 * torch.randn(n), # A) 평균이 6.5, 표준편차가 0.8인 정규분포 데이터
    13.5 + 1.1 * torch.randn(n) # B) 평균이 13.5, 표준편차가 1.1인 정규분포 데이터
)

원래의 20개 데이터와 합치기

In [6]:
data = torch.cat([original, extra]).clamp(0, 20)

실제 생성된 개수가 몇개인지 출력해 보자.

In [7]:
print("학습 데이터 개수:", len(data))

학습 데이터 개수: 2006


신경망 모델을 정의한다.  
1) 입력은 하나, 출력은 하나로 구성
2) 입력층, 은닉층x2, 출력층으로 구성  
[ 1개 → 32개 → Tanh → 32개 → Tanh --> 출력]


In [8]:
# 입력 x -> 신경망 -> 밀도 계산용 점수
net = nn.Sequential(
    nn.Linear(1, 32),  #  입력층(1) -> 은닉층1(32)로 "선형 변환" (W*x + b)
    nn.Tanh(),        # 첫번째 활성함수 ( 무한대 --> -1 ~ 1사이)
    nn.Linear(32, 32),  # 첫번째 은닉층 --> 두번째 은닉층
    nn.Tanh(),          # 두번쨰 활성함수 ( 무한대 --> -1 ~ 1사이)
    nn.Linear(32, 1)    # 두번쨰 은닉층 --> 출력 층 (마지막에는 활성함수가 없으므로, 무한대 값 출력)
)

전처리를 위한 스코어 함수를 정의한다.  
1) 전체 데이터가 0~20이므로, 모두 10을 차감해서, 정규화 함  
2) 정규화 후 모든 데이터는 -1 ~ 1사이로 됨  
3) tanh에 잘 작용하도록 전처리 하기 위한 함수  
4) .squeeze(-1)은 크기 1인 차원 제거.(-1)은 최종 차원제거  
5) 신경망을 거치느라 불필요한 대괄호 한 번 더 감싸진 2차원([[5.3], [12.7], [-2.1]])을 깔끔한 1차원 형태([5.3, 12.7, -2.1])로

In [9]:
def score(x):
    x = (x.reshape(-1, 1) - 10) / 10
    return net(x).squeeze(-1)

200개 점을 0과 20사이에 나타내도록, 나중에 적분을 흉내내기 위해 400개의 작은 구간으로 분할  
torch.linspace(start, end, steps): start부터 end까지 steps개의 숫자를 일정한 간격으로 쪼개서 채워주는 함수

In [10]:
# 확률밀도를 정규화할 구간 [0, 20]
dx = 20 / 400
grid = torch.linspace(dx/2, 20-dx/2, 400)

학습률은 0.01로 설정하고, Adam으로 최적화 수행

In [11]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)

Loop를 돌면서 Loss 계산
\[
\begin{aligned}
\mathcal L(\theta)
&=-\frac1N\sum_{i=1}^{N}
\left[f_\theta(x_i)-\log Z_\theta\right]\\[4pt]
&=\log Z_\theta-\frac1N\sum_{i=1}^{N}f_\theta(x_i)
\end{aligned}
\]

In [ ]:
for epoch in range(1001):
    # log Z = log integral exp(f(x)) dx
    log_Z = torch.logsumexp(score(grid), dim=0) + math.log(dx)

    # Negative log-likelihood
    loss = log_Z - score(data).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d} | NLL = {loss.item():.4f}")

학습된 신경망을 이용해 400개 지점에서 확률밀도를 계산하여 관측과 생성된 분포를 그래프로 그려 비교해 보자    

\begin{aligned}
\mathcal
\log Z_\theta
\approx
\log\left(\sum_{j=1}^{400}e^{f_\theta(u_j)}\right)
+\log\Delta x
\end{aligned}

1) score(grid)	400개 지점에서 신경망 점수 계산  
2) -log_Z	정규화 상수의 로그를 뺌  
3) torch.exp(...)	로그 형태를 실제 확률밀도로 변환  
4) .numpy()	PyTorch Tensor를 NumPy 배열로 변환

In [13]:
with torch.no_grad():
    log_Z = torch.logsumexp(score(grid), dim=0) + math.log(dx)
    density = torch.exp(score(grid) - log_Z).numpy()

앞서 생성한 데이터를 실제로 출력하고, 이를 분포와 비교

In [14]:
x_axis = grid.numpy()     # x값은 그리드 간격들
observed = data.numpy()   # y값은 앞서 생성했던 데이터들

비교를 위한 단일 정규분포 MLE를 생성

In [15]:
# 비교용: 단일 정규분포 MLE
mu = observed.mean()
sigma = observed.std(ddof=0)

normal_pdf = (
    np.exp(-0.5 * ((x_axis - mu) / sigma)**2)
    / (sigma * np.sqrt(2*np.pi))
)
# 동일한 [0,20] 구간에서 정규화
normal_pdf /= normal_pdf.sum() * dx

관측데이터와 두 모델을 비교한 데이터를 그려본다.

In [ ]:
# STEP 5. 관측 데이터와 두 모델 비교
plt.figure(figsize=(10, 5))
plt.hist(observed, bins=45, density=True,
         alpha=0.35, label="Observed data")
plt.plot(x_axis, normal_pdf, "--", linewidth=2,
         label="Single Gaussian")
plt.plot(x_axis, density, linewidth=3,
         label="Neural density model")
plt.xlabel("x")
plt.ylabel("Probability density")
plt.title("Gaussian MLE vs Neural Network MLE")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

이제 학습된 모델에서 실제로 숫자를 생성해 보자.

In [18]:
# STEP 6. 학습한 분포에서 새로운 숫자 생성
with torch.no_grad():
    probabilities = torch.tensor(density * dx)
    indices = torch.multinomial(
        probabilities, 1000, replacement=True
    )
    generated = (
        grid[indices] +
        (torch.rand(1000) - 0.5) * dx
    ).numpy()

생성된 숫자 12개만 출력해 보자.

In [19]:
print("새로 생성한 숫자 12개:")
print(np.round(generated[:12], 2))

새로 생성한 숫자 12개:
[12.58  6.88 12.49  5.05  5.54 13.51 12.77 13.91  7.14 12.43  6.59  7.01]


생성된 수치들과 분포를 비교해 그려보자.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(generated, bins=45, density=True,
         alpha=0.5, label="Generated data")
plt.plot(x_axis, density, linewidth=3,
         label="Learned neural density")
plt.xlabel("x")
plt.ylabel("Probability density")
plt.title("New Data Generated by the Neural Model")
plt.legend()
plt.grid(alpha=0.2)
plt.show()
